# 01. Basic Dataframe Operations

1. Load data into dataframe
2. Dataframe operations
    - Select Data
    - Filter Data
    - Change Column Name
    - Add Column
    - Drop column
    - Select distinct
    - Change column type
    - Concatenate two columns


## 1. Load data into dataframe and explore data

In [0]:
customer_df_raw = spark.read\
    .format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/workspace/pyspark_training/raw_data/sales/customer.csv")

In [0]:
customer_df_raw.show()

In [0]:
display(customer_df_raw) #databrick function

In [0]:
customer_df_raw.printSchema()

## 2. Dataframe operations

### Select Only few columns

In [0]:
customer_df_selected = customer_df_raw.selectExpr("customer_id", "first_name", "last_name")

display(customer_df_selected)


### Filter rows

In [0]:
customer_df_filtered = customer_df_raw.where(customer_df_raw.city == "Chennai")

display(customer_df_filtered)

### Rename Column

In [0]:
customer_df_renamed = customer_df_raw.withColumnRenamed("state", "state_of_address")\
    .withColumnRenamed("email", "contact_email")

display(customer_df_renamed)

### Add Column


In [0]:
from pyspark.sql.functions import lit

customer_df_with_new_col = customer_df_raw.withColumn(
    "sample_col",
    lit("Sample Value")
)

display(customer_df_with_new_col)

In [0]:
from pyspark.sql.functions import current_date, datediff, floor, col

customer_df_with_tenure = customer_df_raw.withColumn(
    "tenure",
    floor(datediff(current_date(), col("signup_date")) / 365.25)
)

display(customer_df_with_tenure)

### Drop column

In [0]:
customer_df_without_sample_col = customer_df_with_tenure.drop("sample_col")
# customer_df_without_sample_col = customer_df_raw.selectExpr("customer_id", "first_name", "last_name")

display(customer_df_without_sample_col)

### Select distinct

In [0]:
customer_df_distinct_city_state = customer_df_raw.select("city", "state").distinct()

display(customer_df_distinct_city_state)

In [0]:
from pyspark.sql import Row

# Create a sample DataFrame with duplicate values
sample_data = [
    Row(id=1, name="Alice", date="2024-06-26"),
    Row(id=1, name="Alice", date="2024-06-26"),
    Row(id=2, name="Bob", date="2024-06-26"),
    Row(id=2, name="Bob", date="2024-06-26")
]

sample_df = spark.createDataFrame(sample_data)

# Remove duplicates
deduped_df = sample_df.dropDuplicates()

display(deduped_df)

### Change column type

In [0]:
customer_df_raw.printSchema()

In [0]:
from pyspark.sql.types import StringType

customer_df_phone_str = customer_df_raw.withColumn("phone", col("phone").cast(StringType()))
customer_df_phone_str.printSchema()

### Concatenate two columns

In [0]:
from pyspark.sql.functions import concat_ws

customer_df_with_full_name = customer_df_raw.withColumn(
    "full_name",
    concat_ws(" ", col("first_name"), col("last_name"))
)

display(customer_df_with_full_name)

### All at once

Prompt: Use dataframe customer_df_raw perform below operations
 - add new column by concatenating first and last name, and save new column as full_name
 - change data type of column "phone" to string 
 - add new column as "tenure" from signup_date of data type int which is number of years from signup_date.
 - filter rows on city = Mumbai
 - add new column as "loyal_customer" if tenure is > 1 then loyal else new
 - select only customer_id, full_name, email, pone, city, state, tenure, loyal_customer


In [0]:
from pyspark.sql.functions import concat_ws, col, current_date, floor, months_between, when
from pyspark.sql.types import StringType, IntegerType

customer_df_transformed = (
    customer_df_raw
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
    .withColumn("phone", col("phone").cast(StringType()))
    .withColumn(
        "tenure",
        floor(months_between(current_date(), col("signup_date")) / 12).cast(IntegerType())
    )
    .filter(col("city") == "Mumbai")
    .withColumn(
        "loyal_customer",
        when(col("tenure") > 1, "loyal").otherwise("new")
    )
    #.distinct()
    .select(
        "customer_id",
        "full_name",
        "email",
        "phone",
        "city",
        "state",
        "tenure",
        "loyal_customer"
    )
)

display(customer_df_transformed)